# BigP3BCI Dataset Analysis

## Считаем количество записей для каждого испытуемого в bigP3BCI

Зачем?
Для корректного разделения датасеты на выборки


In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path().resolve().parents[2]

# папка для метаданных BigP3BCI
PROJECT_ROOT = Path(r"C:\Users\Таисия\Desktop\МФТИ\Диплом_BCI\diploma-bciP300-ssl")
SAVE_DIR = PROJECT_ROOT / "data" / "raw" / "bigp3bci" / "metadata"

SAVE_DIR.mkdir(parents=True, exist_ok=True)

records = {}

# Проходим по всем .edf файлам во всех подпапках
for edf_path in DATA_ROOT.rglob("*.edf"):
    parts = edf_path.parts

    # Ищем уровень вида "StudyA", "StudyB", ...
    try:
        study_idx = next(i for i, p in enumerate(parts) if p.startswith("Study"))
    except StopIteration:
        # На случай каких-то лишних папок вне структуры bigP3BCI
        continue

    study_name = parts[study_idx]      # например, "StudyA"
    subject    = parts[study_idx + 1]  # например, "A_01", "M_15" и т.п.

    key = (study_name, subject)
    records[key] = records.get(key, 0) + 1   # считаем количество EDF-записей

# Собираем табличку
rows = [
    {
        "Study Name": study_name,
        "Subject": subject,
        "Num Recordings": count,   # суммарное количество записей по всем сессиям
    }
    for (study_name, subject), count in sorted(records.items())
]

df = pd.DataFrame(rows)

# Сохраняем и выводим
df.to_csv(SAVE_DIR / "bigP3BCI_recordings_per_subject.csv", index=False)
df.to_excel(SAVE_DIR / "bigP3BCI_recordings_per_subject.xlsx", index=False)
print(df)


    Study Name Subject  Num Recordings
0       StudyA    A_01              30
1       StudyA    A_02              30
2       StudyA    A_03              30
3       StudyA    A_04              30
4       StudyA    A_05              30
..         ...     ...             ...
322    StudyS2   S2_22              12
323    StudyS2   S2_25              12
324    StudyS2   S2_26              12
325    StudyS2   S2_27              12
326    StudyS2   S2_28              12

[327 rows x 3 columns]
